# Crawl corpus ViBioMIR trên Kaggle

Notebook tự chứa code crawl, không cần clone repo hoặc upload file Python.
Bật **Internet** trong cài đặt notebook; dùng CPU, không cần GPU.
Chỉnh ô **1. Cấu hình**, rồi chạy các ô theo thứ tự.

ID bắt đầu và kết thúc đều được tính vào khoảng chọn. Đây là **ID gốc**,
không phải số thứ tự dòng. Giữ nguyên ID/URL nguồn, không dịch hoặc sinh lại text.
Trích HTML/PDF có text; chưa có OCR hoặc browser render JavaScript.
Tài liệu nguồn: [ViBioMIR](https://huggingface.co/datasets/AIGuruTinix/ViBioMIR).


## 1. Cấu hình — chỉnh ID tại đây

Mặc định chọn ID **583–1582**. Để thử trước, đặt `MAX_NEW = 10`;
sau đó đổi thành `None` và chạy lại ô crawl để làm tiếp.
Giữ nguyên `DELAY`, `ATTEMPTS` và phiên bản code khi resume cùng run.
Với khoảng lớn, chia thành nhiều run nhỏ: crawler hiện tại nạp danh sách công việc vào RAM.

`SOURCE_PARQUET` để trống sẽ tải nguồn đã chốt từ Hugging Face.
Nếu đã thêm Parquet vào Kaggle Input, điền đường dẫn `/kaggle/input/.../links_corpus.parquet`.
`RESUME_RUN_PATH` tùy chọn trỏ tới **thư mục run đã giải nén** từ notebook này trong Kaggle Input.


In [ ]:
# ===== CHỈNH CẤU HÌNH TẠI ĐÂY =====
ID_FROM = 583
ID_TO = 1582              # Bao gồm ID này
WORKERS = 4
DELAY = 1.0
ATTEMPTS = 3
MAX_NEW = None            # None: toàn bộ còn lại; 10: thêm tối đa 10 URL

SOURCE_PARQUET = None     # None: tự tải; hoặc '/kaggle/input/.../links_corpus.parquet'
RESUME_RUN_PATH = None    # None: mới/tiếp tục session; hoặc thư mục run trong Kaggle Input
RUN_TAG = None            # None: tự đặt ids_<ID_FROM>_<ID_TO>; đổi khi muốn run mới

from pathlib import Path
import json
import shutil
import sys

if sys.version_info < (3, 11):
    raise RuntimeError('Code crawl yêu cầu Python 3.11 trở lên.')
WORKDIR = Path('/kaggle/working')
DATASET_REVISION = '0148f6f80ffafed5c005af6d506ccfd9d3fb47a7'
SOURCE_SHA256 = 'dee1153f5b9d9010dea18e1b7724b4915ff847b711d036bec8b933486802a971'
SOURCE_URL = f'https://huggingface.co/datasets/AIGuruTinix/ViBioMIR/resolve/{DATASET_REVISION}/links_corpus.parquet'

if type(ID_FROM) is not int or type(ID_TO) is not int or ID_FROM < 0 or ID_TO < ID_FROM:
    raise ValueError('ID_FROM và ID_TO phải là số nguyên; ID_TO >= ID_FROM >= 0.')
if type(WORKERS) is not int or WORKERS < 1 or type(ATTEMPTS) is not int or ATTEMPTS < 1:
    raise ValueError('WORKERS và ATTEMPTS phải là số nguyên dương.')
if not isinstance(DELAY, (int, float)) or not 0 <= DELAY < float('inf'):
    raise ValueError('DELAY phải hữu hạn và không âm.')
if MAX_NEW is not None and (type(MAX_NEW) is not int or MAX_NEW < 1):
    raise ValueError('MAX_NEW phải là None hoặc số nguyên dương.')
tag = RUN_TAG or f'ids_{ID_FROM}_{ID_TO}'
if not isinstance(tag, str) or tag in ('.', '..') or any(separator in tag for separator in ('/', chr(92))):
    raise ValueError('RUN_TAG chỉ là tên thư mục, không phải đường dẫn.')
RUN_DIR = WORKDIR / 'runs' / tag
SOURCE_PATH = WORKDIR / 'data' / 'links_corpus.parquet'
RUN_DIR.parent.mkdir(parents=True, exist_ok=True)
if RESUME_RUN_PATH:
    previous = Path(RESUME_RUN_PATH)
    if not (previous / 'manifest.json').is_file():
        raise ValueError('RESUME_RUN_PATH phải chứa manifest.json, records/ và raw/.')
    if not RUN_DIR.exists():
        shutil.copytree(previous, RUN_DIR)
    else:
        print('Đã có output trong session; dùng run hiện tại, không copy đè.')
if (RUN_DIR / 'manifest.json').exists():
    saved = json.loads((RUN_DIR / 'manifest.json').read_text(encoding='utf-8'))
    if saved.get('id_start_inclusive') != ID_FROM or saved.get('id_stop_exclusive') != ID_TO + 1:
        raise ValueError('Khoảng ID không khớp run đã có. Chọn RUN_TAG mới.')
print(f'ID: {ID_FROM} đến {ID_TO}, bao gồm cả hai đầu')
print(f'Output: {RUN_DIR}')


## 2. Cài dependencies

Lượt mới dùng phiên bản tương thích mà pip chọn trong môi trường Kaggle,
không ép bản lock Windows của repo. Phiên bản thực tế được lưu cùng run.
Khi khôi phục run, notebook cài lại dependencies trực tiếp đã ghi nhận.


In [ ]:
import subprocess
import importlib.metadata

package_names = ['pyarrow', 'requests', 'trafilatura', 'pymupdf', 'tld']
requirements_path = RUN_DIR / 'notebook_requirements.txt'
if requirements_path.exists():
    install_args = ['-r', str(requirements_path)]
else:
    install_args = package_names
RUN_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / 'logs').mkdir(exist_ok=True)
installation = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', *install_args],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace',
)
with (RUN_DIR / 'logs' / 'pip_install.log').open('a', encoding='utf-8') as log:
    log.write(installation.stdout or '')
if installation.stdout:
    print(installation.stdout)
if installation.returncode:
    raise subprocess.CalledProcessError(installation.returncode, installation.args, output=installation.stdout)
versions = {name: importlib.metadata.version(name) for name in package_names}
print(json.dumps(versions, indent=2))


## 3. Chuẩn bị code crawl

Ô dưới giải nén snapshot của code trong repo vào output; không tải code từ mạng.
Các file `.py` sinh ra có thể mở để xem. Snapshot có sẵn của run được giữ nguyên khi resume.
Muốn cập nhật snapshot sau khi sửa code repo, chạy `scripts/build_kaggle_notebook.py` để dựng lại notebook.


In [ ]:
import base64
import hashlib
import zlib

EMBEDDED_FILES = {'audit_snapshot.py': 'c-pO5TW=f36@EA1|1hCj?6M_SmpDdCkun_HL5#$S;W$MBh1iUChvY<i^X#r9Mk3H64}B`oAO(t`2oNVIf;1?aqJ5Gc`k)~HL;WTF&dls`MUk>y)PP0q&biIGd~@co)oQ&v8bvY@OeTdGD_Km}f)rnqf`yWglQb`6kTafy_|M8D;=3Y>M97rbm*Rls6CnyVN)_WQQ2Zcbs!Y6At9AXFjI&e~jH_|RRW7by8>uvAK??SukZF=LOK~?Xp;6Tw<Bw%lKTLQ&iRAA5xrigibN%UyJm8tA*FMVA#Hz`7L2Q+ZM|iBs6n2xw^T)Y>e45wa%R~lgC_-NF>({K*=n%>qhSbE_lq;1Uc-ZOF5)OfZ*;8V>ek~LusNkVXCbfIluQB{nq9|3O4fdjxhq+$$h|Vv>Vd07d%4IShIAt;F{=#Xu=hA{S&UjF`M7SW(VUG7vDigP1y_cja=8^nZxXwFo+;uukGkSRz$--?4&L`X@js>AM1Rj<*U%O=}yoopr<yhp9Fc%PWsIItm)D;`l8Q|Sf<1$8SJ<=;~i0~<yQ7#O8KTHGPZ#P*z9)>=*Jg(F2rey)^m3Uf8CBnfIRf?+_Op<nU?47nf&BOqC-eItnmr*g;SYNjSz?H*PD@T8jq};XZP>p3uBbQ!#c(yAsPvi)Gz_JcAQLZtkdhJn+9ZM^Xk1Lv-$5|vaMdY+i^mnI)fYvs9Wl)RWv6Q}vfHSln)-mET7wlI&D#d-IpmLq}G_8&ri)9}3BA8SL7&$O?r;&PjTNg0uF=1DkwmdOS9ubDX?4eO34l~p^P-4K2)Q%%Vn9T5{RFN~($`mOxN@muyv{V6^mOZuV_--WJh7UA>GwiUDjDQ#UV8<oV9p=ace<O)xtTeaV!?stXem2#w=l;La9<5<UuCL_0h^NC;<Sa=Grd?fa<dK1b%JWJDsR}8_b_^OAr)Ng5f~q<#EGb+&j?!J%`I*h3%Dbrjl1I<+traq>H*UEadt;!DMB-XfopSVC`|!%mKQ0sKAuNhSAxIV-8~}hE&{?Z^a4+9t2VB}SB8pfr5y4&tj4JZVHv!Lu_RO%E3O>~?y*{#=SlnB~Jd!|^<PPl(atjiJoHF+gUOa^laLD<2!HY7dHE%3{6$|g%ARR?~OcXoA*<2AEX*qD%`2^Zd(@pemHw~w-g-$Uq6yzW!^1O;JY^gbi3pIaHjs*)6r$b{omterkAdOm&`N2gb=+#*>Yh708P>nm<$NJ1C13<3d4n1A_!h(~1{gU=P0n5u6L}f)D@DLbP<OdP}yzBh*(Y-gimmIH=CkY6MO^D$Ah~uR2E6{S<l&2QCE;AQOP!`e2?$LRl7Ya-|0y>h|J0x|eLbHeiixxv>!!J!X3!SmlT4~9i1Zcr1xeqhQ*S68>icl7*>SpM*wCi1A%!l!LZcTRT^Llt$C2C2fFQ@V_5Gwm0)di72FF2_C5bJH7Bpu>yuQ0K}Q#UHY(^+H<bKCO`m|ZOsM%7-O2filrg9(mo;GaP-{ca_r;h?G%YF~t!%h(64W^?eBc_SiZ@H=TDmTak_p+Yg=C|fHTOOkVLg29%d$Foc%p=(V7yFV(UXxcEYy;QMDvK1;&C^FEeydm3JA6{jP^`ky+5eM|*wI8h{?~vfib-cATO|>+_mje^?@$&`Bc)U^^nu<N0*4$*+M2V5pwzm!VxsqnGoSPiKKzrrJc~S6SLcOAo<H2OlPN~J0B;Kee#1~#JxC$n&az60fEF2x9TRpsed>G}2x7+UR-cOEqZr>QTk^Ut$sNREBnjww+pfjY+`fTI1b)7$}J+~srSLBlB)>UO&<=jsL<(p>MS>fv&>_HL=l0bLA-6#$Au#u5aKs=xd-F|TTuzT-5?65@aqp=^0L?}*&yk0FP!WIvy_LghW#T_+cH_rrf891hI#WWK1MUE7Y^*3BZ_3~2Cufad^LX~DdL7%%~pUvK4#~wTsc;|z}B2wWI?q?Q?ip50JU9irynsA9K!j0ZbTHwAlIr~>KX2ID%SuW_74E?X4pFBfT`H;&kwkOX|KY^ze*RdF%pMDyU-&0n#6kfY=tdXVj!*CAI7rU88Za<U(5H}qa59mIgh~oZX7Ntt4<2X;dCMxu;Fkkzxe6WLjk)z|jezASr$NhNQUNCJPEI46#%BG`{giBK8$6W1|nU6f$TQqceqeiWW!-GI%$VBH#Z7_<oEeJBJ9khAPI9}j{6K#v0gY&}DB2QlJi5{x<mI|40%nmx-k<P3@9~HXwrN@zeoC~(Ca2dlTQ{CH|oS<E53^BJS>Qt*HyjrOmdMw#wi(wU786R~kM^Y2g?YVO)Lv@P@Kr;A2jT+Enk&8L=*^png$)UutZnoN9A{T(EujRQ<J)AkU!G-;~feFasv5Xos*D5-G7d-Z&8XDrBuZ#Bf@_L%bXMR!3l9RQ=*4u2Gt-pYtJV-MEfzF+>NMjAO2+kT1xH+2J(k`O=7ER95D8zRFgr2Z5o8z9dMEs=0)!H5kelvsC?RBTRj`9-ysc#d}UPZkFj0(*m<LXI5SjHJ935KYHHassCCMbCz<=_ntFzjHjbhHdM+qAqeL|CTLlv6c<Xee|nP&X8-Wh&e3=Ni`M7f0*sRr)I!nb*6kF5Nn>!lFSFi=byuOa~E<cS8VneJ!mA0^0MK8WJ>^nWXnvf4Da{(SPgmb}MlmT$mCwd$OD~)#{k8%F?teG%{1!0jmcJngP=;>&|*8K)Iud0gjlH<ZctM;qX|uU(df*7nlHv`EA-f68_Y8-QMMUb7$H42ryIdbOD+%Nu=U$Kpdt_xxemXva{J(q_#s(pcdDbYKU>+3V7rvWxR_q(glpL5{<`%n!kakba^|@teFyZ3>*EPap?agVRZ;rCoK7h0_5~x0V>gaQ(s=U`+7<qaWJ&@Wox@1ivsg`x+95V(9$ySR)=Z(4pkctDV3FZf6csG1PsTX@N^YE8sV<i+qjh-(!3l7c1yX1vLDN&w|O%=+@eTyCqj-VAoFg`LMWF}ezc475vsRAjA<l8b|VOatqen3?5wx3305nHz5|Cahn4Cml(>rWX%D|ngj@8d3!QjVh%SCiDB~pWZHyGd|E)34v<WkW^ypv$?{#xrH$@N6-2=t5trTb3C`u3XQ;#Rpg9%#PEwiPY%`MG5I@yk(B2~Q`zr6M8tZy3ACC-^_Y(JXF^OJv<%^nT)#)`8qB`y-@r@t#}c<HZ!zMpMBJU{uaWXafkOyt>LF$xrbpPQQ=yL*26c>ytJ&%l57rD^7h^V9FB1xM3&_Eo}mNjU1M!vam6P0mlgHsmIp9{b@BXJ1Vb`5(XM)$$(fEO0d70So6Re~F3x8xBi;{Qms(3(0=?^z2zOVPkpr%<HdZ+s^RP0Itsa23}yCZ>J~rhZ(Q>eWs$Ie61R!QLAqgW4mw9jqQ38#QyV}Ke8jRyB83}Ocy8+mH}QgR2GV8E!*y6;KB4bW8hE<InZE!b3$djx=jvD#^w3xXBd9WJ=w8tLR<Y=%<P1ujbObBHA)yo&E{QOHV4IDi+J9a8#9Rw##XwUM%>i_!tllDLFLaaSyXy?z;apk8um<otxH=Cgs9tWh$VDbJZ>Au1FbI{w)MS;M6HIRIMhINgS|B`fYkoBXp&Kzj^TZ)i>dJ1`;Xsddz16iKW9}eFbHfmud}tIMwdJDS{JI$=`tE^?pKxZ!>5$HUkA<ld+*IVU5q9dx*yVi-#UE^;=h9z>5m^HN($ZQ8PmXFaDyG$Oxt;0dpU%?SJF~e>w=~0osIg!rpUC{E0SH*CoY5F#NRicJbb(1r&WqB_M*et68RNQ=^m?gd0o!!*;2^bvb8~m)i7be13D<K7lL#5>{$S~Ir}zi#v?iVa>@>ADFNr-5|$~zal&c9K!F6qGYp4GbHZYcXgO+$Mdzn~qT8>%9V2PJ#V6XK35D+Do2iXp0v5@_5{q!&Kk%XZ;S4x{)XhETvZ2R>vtO`d_8uu}?);djk6E|dW%_RqZ-!N>S69gqD2~wN%|LUc8+?2-$2Z<~%+0Uy3y--TdHPg$>uzl>Z-)v*1IdS%df*TBE<e$pd~k1zeYjAz)jIwCeNY43etj4K<xdFKlL;@kK!$z(Hfdr#(WV#V8pXzO!v)m1^ZvaDPyEOCAHDndi5G{fP$LenXkRd1e}Vgh{+-wU%8Z`X_Y(|-d|!9lzE8h7`+jxv$MKD+Ob8o20+M7T#}MSM0xkV}_4RB21)<}oc>',
 'crawler_snapshot.py': 'c-qZ<|C1cWmFMq7@IMqH9Mugyr5=6ZV5G(?gs>HW;Dqqq#i-q`>8_qGrn{=Cs-7>*%nx3N_iUfJFt^?f9I+b<<~G)0uMNA$`3|&q{y_x)haxP%|HXaYdzo38)m1$s9T&Ikdel{w`7-n6%a<?Tw>)#^%y;7F;`HU~H@#sN?oHRyWP0Gu<7rw1-i_3Ia}i|+-Zb2brg_Jkr6c~g$ig^@lWi}}yf_&}`}iH4IdkUGN5)w?^9F<QqF7|nVBp2Gd72eom?UWt7IB*7k3RC~Bl>N&JrA=yQq_j(bQ%r0kgj(%T_i=6b-YnD4j0p66b}niIvFmqEJ}(%S~HbyO|mE)y_%-eYx~h~QKYCJ=7V86n@^)68kGQwXup`oTe_lpGYgY&JIYkCQCLJpJd1Q8{R}MN?^|i2yC2QMcp5B<cxrmWqoWFgA|2?8X87B0#dGs83G)fMVxHw6ya{Ge5u$Ebewv4)5yqRFH*e%=VjgB;F;QKb<EJ&gSNV&2$iifl&dSHKU-^UvQ9h=_ov0`uX&&<;9+u7L2j%aAkq>$(-`HbbB=HbLB;(;eF0v_&l2N5Qig$0Manhk5`Ft7|ZrLo|N(-)#=^8ib)eG@7dX-;WL^eMNvn<^M1ppn@w*e8wqKNqGaOo@Bzyc`v>zfv;@G>u~F=yd8o`xVoJ*Z+j(mmUbiotj~q8@$c+RfLlzjDLtdH(JN|ItTYzd}Ft0=yaJQJ4)US7{wa+1YNejxsl2dG3{4gDc;<cH`EH!ajk2^bxEQZ&3_e?Ttqt@$g?3p-+h+*&s>xTAFG>sl#><=jk}jKr*d%31Sp)V|*=|qzWWOr9nOk&s}^<N(7T=U)8W+03p$UinoXeHJH7)m@pa2+aR4sNz2c+{5IiutV-21$lz2f<BsP`2mFS{!y=u<!@<@;5#`)c$Fm1RZ-C6QALVPBa$+g7ol%^%<RS0f$`)AfU~_pfNO$;A)hVLc9H6)=4Q`NxvxtEP^viqF8~eebSrh?LLp%b|R{J<8W^><y6YOO%ScUYO`)hs6qWLr&M!LUxq*8^}zZz${^@bj14Hl?iFj~y!c?$(Gq$DRk9p=M0?!6FBa}1LhdQ$YxwSy?3wc7F*#dz&$zil-;O~X;yYK2I+5h{_ltf?jp42Zq97%Y&{s{u@Pr>e79f%%HD_<lf4qGgRKs~b@pli=S@d=G@~(S9X>SB!3wf}Fr>D&uNtZp7aLD#ddQus$epn(jrJO@#?9tZDpK)S_W)VDuXlV==;@Ea=~O;i}(}G77Aoc>i{WL3DU1&RhM?0iQ)dzSwGI{^sq%yS?ak!*2gcD(NZkZD(kv`0eEIeCJp|n&t%0`$1l0@x0Y$MD7zJ2~Z#?;V!h>x=e|MIl78v)WT+OW;0zBdi~Ms0MDvF@z`t8*R!6FZ8#odVFxt97&0%v-0(K;eHdfY_)a)oMAyKuvzGtLq5wbl!YPs40kID+^faVP80Ug^8Ak4KlBReFy(HT6*4EH!pplrTi)<J<U6favQ;ccNzZtwL`vdJeEvEp6F&_u{Z~~!VAbSasO71<@^EUl>M2H0u`F-jLL?bYepg5REIT-RFh7<z%qg914j-%<QMYY>pcr_-j?av`hMc9V~oEMKEhK)R$AV_X#odmAMEXmEV<54c-lC>IcnPGKw*&g@4-<E&OU{Nzj7Skz&EC_5gsQED9-(cDh7LsAnLYZ~|Ic$Ca5&|BkVEy!N<tSc`^IQp<qXh_Uh^<Ud&-Lfau5(+|dnGZp)cbLqfoC^;ecXHf=1X3<C?+ZR=z$mKUcMNQ<6#UA$YegxQfxiw!6k1OOO$p}Z;XK+cv}&^;~^@B6XNWuJ=EC>@rsrQCJ@{?8}JM8trCYk_x8xC9!&{qap^$Hwz`<=6eyd<w8L2y6yk|?VqgeTSps$d5FMJ~wx%mA>>0ts(*EwGDCYS$H_o2j{O(z11{RC6|67`cv41_`(QC{DYa8Ansy~(;>lx7hL8!#xfH+(>)x6;>8av<KJl8Loq+La(XH#nSy(fF#`qDu2#jIsdX0S{B13PGYm%a6Fw_7t1yL=6FK$WEp&g(tPEiAzX8p|?hXeF^4EQNH^@TiU<(8s-NZ!W^=8tocjPt$bB$03o3vWQkm7H=*19T5YS%jN4B{E4<KMPhF*aFham=`0kGw7C-<<XEQ}7P}fiyW@3LyL*$Ex`MHBJqK-OWEDbGQ;7fPI?GhYuMI{oS0|9!%&A+{|G7ZRY09wtd+N9n+~j{0OOB$E*~TFB@1~e<LQHv9Y|*;WOg)N{8V0dy!M2wWGlcZk(JK|_x7Lc)+Hsfiy7RHR@E&5>(12DfnmG=@ED31T=(29HX*`Q*H+Im!G~>jq2_Q`(*Ql95`W}RHi1ee@A?*(v24T$bHauoezO=z#>Bn(pSkN%kkq7G;XguJxL=5=fHnt}V2kn|lV^y!Y*ShDMT(FN89;EMzv}D2eJhcj2nQR_310-*nDKLwn6h;}V9S@7RHw|Z7qtMGXyi9Gg3Vab}1@ZNi&KfvEghi3@8bFWY(SW~$R&L-}2T*kd(z*U#%2`~E_B?)1TXKt=&~H}qy4N6G#A9rODLB;HT0B}4aJ2T-BS5vH{o!<x$GcIxlrqMiX_>YtYRhUWkESBGG)W@ZkFiFvi?);?^=!}Y2uKWGzr5@V!rFaj*;(yW=j`x_Co+46`4o`pz52MEG2O!xX3bhT(iWq*i38Scy3e!hmVstikUg|O9rKPrXHq5Qt*!AEb!gxp2lmdaH;qFX-iFt$aC%|k$CVnvVooAYOEXQ3P~oR<;X}IAHnRPxPtS7NQ}VumVQGI#lP6HsE><3Mb!n!Cfyif43zZJb{`-U=Oe1)@)}TQ|oeczUv?^vi%@9=a!;wza=Wt(r4)@*9RwG6z1*@Rex~R)rH-6J}nf;jGDp>^I<cAGOgcXux25wPXis@)HRm$*WDQL%yICD=_F1ChPX`rJ)SP-ux9TKetAT5vRjeq6(w-(vr7Sz7`XSEWF)){oB<9M4&4AX4B$mtz#U-XLg#J)XBnKpcbxT97y)_9&ycO&eH9W$>=zq4WKk<<apfkwE1@J(%D#`J3OYk3TBES?{c7$)d5eA(%uW~czCco%n<S+HiN1H;$&TRNwnLma`A5l#4&a0*2XilmoDP&?eZRbe4$6?MCG09;gd1U>~1vPGiz1dA8pxV{^wi`+T<nhI@OL5Fw5r8oPokVMK>!xbuAE}b%8*KM4ozJPA9bntt8<`8v-haU5Y&(xlG0HjVCP@X0d95@$Ig}-t;XuMQi0Zx0V0VyY)AbxFnL`MPrQI78VD?gp)9Od6+^~3e&&(g7T7Uz643;d>DkhT!o(Qt1-8ZXk3Vsky$?RuAcP6;~L5zQ97{JJEgov%F*0hEh=Js{IYf4S0D3Pnnftrfv2B855G9$Kg(yR+!z+V#?x(#;YK-u~<c718$_gt8{|(I9ArG$T;t=^;7|{ZtkXrTva+ob5bNO^_1B@0ah#aRSyL`+W`a1mkK)1pE`r5T}7!fy>qu9#6yV0UiBN3z7y`ttkf1gIUI4gd?10%B0GJ>av})1>1mdbeSIzOQPBCD6l6ETR6jjj;1|g^P1iqMh~rEmJUHkyyx_I$Ekt5yop&1gBSs+)$|P3_Kf<XPbK!^whGrZLJV(dzckEuk*F=Kc5J@FfQSV0pt<oJ+PA{iSsym-5>0{xx~Li_wdpZljj<I?^i0zK4X0tAdu$t$rE_h67|ltypcy7<W(U~e;$kqsA!<5i2^tK9OsxeBnLX&9U8x+}NG?GlGykx@F6MMR4GctqK`c{>I{kWqRAy1u+!Kt8x-kt8dg}osD(qb_fq-_;FLjBF@-Hs0e(MT)6C*?^h$XZNr0qq>`$K7~c?pJ3=fpt8pt#w|RWk|L3y4=eu~jl3*ahSjiQUH?>wn82!>Q9UzY!(6MiGG>V0%?4%A#%37#;8t#KsD1jLHF3PZ?U>FSQ}36iQpt0-Vm$;GHhySgjx-?t$H;+LpnSS;#2I#1atK@v9qkStw9NSXCj0)!;SNQv0MzF#m-k90oP(IrLwLNVj%n8>|kC<n=4)!j&P>K8vH6ud|b4Htm2Pu-Fa|XXm5wr8j%s;F-=7XP@9-7yaY*k|lDvagCclW0Uw0a@ZstmFUv?;#Nh?Doz2S=_$=g4mzcXX#XS8@xuJX`#j0ltjP7-rR{)A+x95+3XCVKyqaSL?xo9w!JDncG5lDRnY<m=_KvOFx^E;#cD$#tST9^S-+mzA7!Ycc0((R&kigmvKJHz?WF2@#s<qD~1x35$m!daMxT??#QO0{bNoT=mcS&0hzCVZgs?h3FEvACy&sa`n_XZNA$g3hw#%Zg9(2g@=RBd65%hS~B>6wv}#?#$)lfqs*a0KyupnkTfKbsf&o83NH<p<l-bSs=TN|iP;7fBwDBMr#!R0+aa#A=NVQZg3^FJa9VtH(6gtsV-P_8oe*zM<oyl_EC>u|SN${0BMYhNPGt$j0O;opz;m$*vp*XV@%&SCPn`4EZ#Q<}EszcU@-85;0X{L3m9{&r+a0g~b&dt-@2l1#6HjhsWMh?HU2a3_!ETHg|#XK-IDL^jc4MAh);KG!un0C4-R_V9yAExwckA(}Q`OV@X5}lJRh|NC-Ox4*gR;fnl)qMaOLDL~*5uWU!*-R#I^5sf!oSKh<t%ic!SPJSi4C2G(va3=)^}&#u(CHdAwog-rS@(&CN%m&wLfOu_`(F9LV$l}8XTQ9gV4Cv>MWhad`>-e<iF7azEL8i#`sb{^;+qE)TSL}}@B73#F`rS`LqK6RNXlL7<s8R(Q7CFp4}WaqcwDl|3MQ8EGXLoe}s%x4N!TY)8$gm#;dzBP1b2&MEW-J?yJ)|f3|@jtYhyA5?k_d<dqZ44LAJ=5_ncF(UOFjk7&gVnn_pKfnPSfguK#-Lf+E4r%?UcZ`Gj7&e05S_vcio4QY(L%(Mv~?^!p}muPw6a&V9)zlw0sRK5XY-cfbo_Z$|E)2!%0YCzTR01=pRZsA0vmh$*gi$?Nv3EZYl|~)5?AimFje^H=m9rq0+a++w?^C=e1f0Wa`d{M7}~DeJUVq9U9lS6ZI&ee!90bOc~EY%ju_Z10m2z!n{9z(|8TCvi(}aB@sLCXw#H-s<Q>#KjtIyac|^6P<T$Hlo#?@XBI9RI*Mn=-St%bB`-Q(;2ghT$ld&_+p|YeIFj&nju{mnz;~I5F;~G*Pz{EQ=(B^$`z<VH@4Oi}WO}to;W6Mvf`3@Wq&Ry`H;D1(cMiys0-(81!w|kEMoToqkhW<P)T0VyZzNDN5{F4@Ai4NsKTlP=^+$@3h+b40#Iz6hR8H6yiQ~y~{yFQr7xO*D4M$^_a9V8?xm0-`-8aD4TN)xJ3+NAR|L?n)J1e4X6rRyInY}lRd1WCfbOovBbd%kUMkhR7b=h<QJ@?%i;v0kyE)PoBgvW~G?!+V!K^1glR<(E8i`N5QWsp2)V{#Nu~co8sAag}bFm2+Zcanf#;UVkM2<SVxuO$VEN`1YCo<3|J0Oe?ZO_X5t7>)lmI`}&5p*|Z>*yM={&VYNupIFG<atYllI9s2gITd#VD0{_@=>_g*e;FM(ZVC`8O!KSc!n!1)CtYfCt@hPYgF^t}kPBnPfFocNu%<B4~Rx*<3M(Njv4;^cGZDq@=401Iq+Vr^@ra{u40%m)2<097P1ED}P9VG%7=xWWi)>N`)nE{gax}~_Bz}D`_ZFW~yKCfjv?W%82bsjVdYkKV{Z=xERb%}Z+5d;kr=Uqs!rw80#(*LQ&Gz^mg@9HfpbfMPQuz8KdIiHProwzGl<gK1*j4BWKrQIC!;;!w!!a537NpIow)~V$TiR;i-Oh0s;s#K)ALt>$l2)THU4JH|}h0U(LtW{K%G^TA4Qjln=Xvmc1XhWRs-g-R|PqM0E){=joZr2T@;~9l@Nk&?TRgEpFPeNK*imLls5UU&qsCA`wFZ-}wRn|Z0t=F+sX!dxFT6{DTgRg3m(OrJ=yOn~Z^j1xW-`7WTZlnDmlIKGSiAqFN|ESYL6UV16Z92ByevRp+dZsg~ft&KfaA2J4UQ^6-H7}NS>v-k)MqWv-v}%5PJ$G6kXxGB=2FmH3f#x+XpN7{Q2iolq>tR}&!E@ddPv|Z+>NuL@+v;`??VVC%bm`AIR%p<l3nHxRp<h1L$RGD!O5xH<t|w4g>yt0y6rkzCy2sF=*n+k@MA#*C9&FnbZTBJx8i>)=!5WHNrH8Fw&o&AkK&uK}+Ed*JlTS#e!74qR{`9a(y0z3jOw<wBD959Edm$WEw-zJxrAci0x+tLLWW&5eA7`vpx#~<)>UNnYcRbtkpy!<NuF97+(!&rMeA`W^MkAuMkP1fcaA(8YWj?#p!4J@_hE8?`ayZ3|?wF$ro0C*$)(X`mEFjzTFO0Os8bW>Ju}G@^)SN$$mt9r9{u8ZnczDP|EaSR#L?)i$Y#T+2M`?sCl)_s#)Ig5w0GfSPYMtKai1&C`O;x8-7AleD;M;+i=}d*pO`!rxObi)9tzQI<Yl9dcZ3)L)r0qd_3o@m}k;D&>;0gUXG~V%W8m+-Ge-?t9Ork}`al8_^t*8o?U`ujfcx=D++3+}ucbOhywnDD+-*f!;JpaAGe=qXir|56ND%)J5y6GoFByYvYKemOPT?qR{D}e-TQ^e!mbbm&1N;4XL4iob(1#H2*Hzk)~Ob6m}UZH@Uh^Al^Q!}Aefp{v(rfTpKu(l+*S_cj4K>)B4MN?}%FW;e@+8=eb_OhowSs+l3S6n_sa9qJCDGsK*=;Cn(6CM3ZQtR_BuS5#X^-t<W*?^iZFahov?2%iUzf_y%Sg#H%aM0j`MglagtY=EKT@e$ZP!#!`FqjGinPM)n%tiQhKu#lddPk@2vS<UA3uap6Ft>W_-jV0GoAG!W#7PkunO>t#V5Jq>sq8KS#|}4joz(0t!Z|jff2_P><5f;8yS3$i<JIS1SQA$Ozv9&3i(7XItwsf9;@xN1*v1v5vU)2)+RAb%eI%DK-okuphXqRfs32^PBnrekHcK4PD-MBXrbdqUB9urt1byX*h|#!T%SaFX0y-yhxutFyzgON-<clDJr=QeK8#ewqpjGC8yVh>nVSP0rkU-%yqOd?Pshap)E2k`hLt7_=jDrC`(oTZdC;lv+m@+sX8J9mc^XZE|Q7PZ+=1SPQRy?YLGzps<=N#puDyP+t-bocUR_#deC|Jzr92YTgoF!fIG@)&;W+cGDPD>IjoV-Ya#V!o38e(ou!U!;C_=U+{K+c=wY6NbMf2Q8Io;!Db{hW{hsOjoK79-WeQ1NO7OT{&F;sXRUs<*fECwlnb=It}L{r)#wo45DY`cJm~>PkY*>!n9DAx72Ta8Aw-(T-qu{i*X$U+5dvG6$eDgWm=XMt<ArfewL3OA30hou!L81&J9MZEX6(+O8X%YDBAXGyvv!+gpS^YHfRKhyK}LqaXGpTP>o{`qSO>U8OH0N`e2~e4*9jdRL}_4EDk-k!tLou+#yoiRHuc5@?McU7L5p45n(E5$mJu>k=aWJXcY+$(Pb}XECp}OfGepSW88)u`+c8n(daIWOmzQT}#Sw<O9i#(esr9C}()kd`G-EPPUGE97ixB&LL7snP@@Yb9j?>%tcB79a5qkLezAyKeLruLG2^rEzZMZD$(ABG5{9cG}A13A%Hntvt7q9%R=;JiCel{(n2hA%EcDqt&CyMfTF__M=?muwf#xGS!zZzl!2xhRI<p&opl1}yeN6CaXu2kC>%}@#}sU*yTNW^3rVbEV6jtY<^g@!L>|#mn_hD8t%TECL}U>%C_a`gFq-lv5OFvlfnXP$77Nroh2}7Zvj9-KHb-xH1BX~4_+j5!y}`9rh-lkZ?j}W+K4@!LYT5jOTau_Y86M+vy|NZnEUP)cNKx`_t4My-XJSd#=4>5$+SWK^cvbbI81*EGO$qZ164mIXEiCZ|WjT;SPpWp=t{+#N4`c!913xv?t0eU3cxDOoG(ioh_q*%<O1_}H$l2n_EWStRq%M$)n0(|OT3@C2E5WMXV0RseOL<CBk~Hf<b-v{+7t|45|1@2XXC%q=HnmPxlTMxuWqD5v>n%O007{ulLk@~C<Jq>~wrQir+$Hm&;|~Xt{%J7AQ4px#VNIpY@L6D#I#yDmx`>Mnmqj=+J{+^+ytZRR*ca6JaMYCuIMopE(S-A}Du!pNj$PWg`>Gm_sHV!R$dUmK$WmjMsssv4i1sSSn`-S+t&9~IvYRwTA=Cv<)m0+1Ik!?G*p7Iy^4*XtO~Jvo_@GU}9j~$tvDQmP<<qz<MFcSuS_RgTbNI#%9ljxI`?nY8FLb*;IRgyo4_-Vo9*-sC&rloU&B2UK{WGP|A%y^o^aKW#kX!{;8?xs>m!;#~aN41eW7Xa;Ap6vO5~?unvW1$OWFe=4<WBaS+|8b&?C2(3SeJ!fwFftczUet)Gzr=|RIpqyq6`NKon^|V>4Sbr0J#Uu^}1SKe~lFl-h|wuJ<9lZ39>svB~`Q}YgpnfQqhi}gia7i0I{661=0y-fmty$3D%gQ1`f_y^keE`PTrafs68cgvQ51dvi;4^esK4fpS^SU*LOeu{Qb{A`s~l|{`qG=yZe*7KmYu_&p+(zoJ9?wcVic{!#nJP5KzE6(yU@Jrkw*PwbuH%E@RBpu(~Z6gayRO2|;9P7GDneMUJ+<d(dF=Zmc$w6gLAUi<z1>p#rbn?s_8#CG+HqoT9-k7^UFlib>46)Y|EqwKGwRr+dLHo=GijQ-7L!qop%Kw5YTy><9#i%?vU+Dd)+^0m#`7EwFTp%_6KqqSItPOLrm_O4^a2X?)aLO_<N%4@tI^QV*(VQv-y--v8qFKfeE)pWgrE5BJ~s$^GAb@W+37k66w9AO8N2cmDc|pZwKVAO85upZ&#`KmR9w`_nsL{`~Lm|IM$y{PYj^fB5!SAAfTHzx?W}4}VGDzIgXPfAQ|$ef9Q-_y6+y^!(-jdgsgE{NnzH|MJz3eqFQS`+xl8qx*0F_5EM`>i+NEW!V4q?fXCa{g?mz?pGgweE%<gc>i~w(A&>`_Sp|Ggk*B>)8Ed$&)&Ir=Lg=_-G59F$FRUF)|y_O-2E+H<agi4glygW^n=u!;hR0vG?_AJJ!3q9<tv2|W+~?z=WZ*7YA54KsUauL2QPbp<2voaAg^{QNhbC<b!nPeScOlx^`#Iber8>D)NHF;$RgDU3W}RSLMEg^I?O6m=q!(Vp`saZQ-zwonq!i4^3Z}J%LAtHl4PjkdDky?PvARB#m#jq`+gO?NN-E(Q5Px~Vs17dn=-_T1|-_5l9%Hlg!FV=T&GZcSaXquYzp;9b~MY=HCB4_uR?xae(d%p8jHtAm-)rBN0)K%dRDP}r7>w7cgl)?;xa_ri#docG*iG9WD1a_evm@P|8DD>8}KMQg6Gx|2|I{H6lbiXHx`tW3E6Fl*B()ogmqME+@qsEz5n0;^tZqK<Ntj7{%_v}Z&1%){LQbw`0@X~|D%tN3i4?1b|!Z}CZ=}p&R=2$-uv|Z_(;FH`;W*8iWUC%2OgJg`!(oux~xPdEeR-Gu*6Aw73uQCJH^62ti+U(<se<K;bXEu0P7LlYll0;B=H)ljwnF@rXxz&fbvGT8w!hNU@%7SKrlc0)31N})ra5z^1bgL4Z}ISP(pV`nx{mDfd5AwB=d!vhhOHfk=W5v(A3+b2z$z-OG}jY_NlWW)<7rD#`W!JCDT+Ey&;jc&Z*$AD$8ydZ;py!y&<qE%>@>tQT--W9fg({L{!)?5=7})N|dFz;fB4llQqp|l^dGJR-15`q78}osZ!sRX(W_u8E=TyJeS9%xGwHf&?~}Q67k*QjGeLetIU0=@+MGv94h4Vk-C~G#JJRKa5PeBRrQH5NaRqOn5{@$+v_8%Tos~nsZreu#38Coxlj}KO+oRT8=&W89&iiT>0c>~o+W-=6Ke84Ip-Nj<c+uemd#O}T%;-}UxDpNngo+1LChsH8f>rIZdVU-CSu-XL+I&tI|gP?|KR*HX&&!0JHSisK~Je-4r*NToLmuAU!ppP>19&@*RkDkYJu5pe|?!3R%Ki`Ri=f<y_?ztgFR}16A%)+%bcBn+q1cW<O7dcr!t<g3ly6JiSgh}G)+x@$5Z5O&||U>$_g8X6DbvUiRfjYEO&s)*Q9iiReqQ~?IG260I%@7t%VDVkIp1DaQr-b2tlFA;x-W9Pw}LK!7;Kw;>EGAoZH(ruh5_|r&<NvQ)~k6ZgWbEpxh&S8F?zSY6A3_OHlH=##~0L^Tt$Ig9&@5cNO+`%EYuf7lfNZrZ!Wp*}P6j=0mzAU3IgP38=q1`I_V|WB!Dl2^-<aO5#e=OL4%ee~zV0DOrWeJx;7yj$}0{hmg0Qeh)bH6S*epbVlMGx~vz}9+<f<sGlio9tp-_8His~dk&FVnM@Do8#O8K1J!@Bq#7e?tK4kJ(jckY#o5HQK`6s*`}o*N`>p0e_C85pB@<Vs?j>fcGYqj{RLOTuGpF&qa-wuDcWFk?)g^CX5)Z?v&l66PW*p=`m#kC~rJe;JBCixPw@cZ+btg2PL%1VllVv6%aY2wrzZieD#JDOyx1*#|$|$w1TRJ)`_Y?KQF{?rZ<skeL{in_Q#8yyUy(#Te9_~hNZl|(NjmH|5ZpiSYzCCP$r>o^i%{p9_=ZTxEP4h(Cx0)%{r!wJBdsX_Ym7Sv8NlM@3bf-q&t|r)|&Pn$)kvW-IFCb4mk_C;&ND`!~wB7W&&>G<kzGsI%8ro66HaF~&nw~`LBg?b=YH_j`iy^r@M;|nlf7?=~SN3UHz8i#+2n@*SnA|8Cpn4Fu7L>G4JL;wpP(5cmMbh*cIW=iHnv>R71ZjI=sPR@D^EvZqM**i|5mLED9G+m`ThdakgFP=f$`kyaf)I#3cqw~|AZC+ES_>x)*_I_~>NmV|7fEN)HTS&G6MMF(nXbzMyPV+2j9@@oW|+2iwfp9V9=lIbuA{}&WI@s`n6rfTtf4V1B|TE=acmzgeW^@vgA%UKDM>!1U@+Uca=iMp>SZq+wYx|-V76r)fZ!eZ%4?i{2d!tj2(8cQxQOkS-_eel^NNirZv#{il$QX;Y$%A>=LM-tLUv|W`K};UW=J^6<1bw=?K24z4OpZ^w{TdmZ;>$XPI7TZ_@ufy1N+LzQ8AM>i{HUZZ;DhxJdIWF3@N~y3S7yS0_)XxMzl>#>OFEC?zFn@p#wL;1f)D9zX{fM?ZwIYb5)a)zNjYO9cIO8(s4*h#snMNRmaG&JtetPeShi2!Hwr$y80rmk4jEq5sOs>L~{UyFwdLd8<0sA`@Et~4MW+>J$BY-a|y`h5{^5}t5x<}PefaG$(Yf`T@$iU!)z5s#zQ#jp9UdZ5HmxxD%j|#cQ3ti^~J%pe-GbA>-olW|C;f!Jj4UK>A>QVwNd>|WcWa&_+Zra`i*+(st?XQVUk|I;asSKEJSd&TRM?Qet6B73>UMdtFlfBuVjY*FU~e9>uz27y1cyYN|tO}KHayj6Zv{1)Bj%wFD-mq;;_jvRn3JuZxVz#3sj>dJZ}>GGG{90H3>yadADqWQ<_@Bu6%irbf&BuJWW2+s*Jm_DqCnZ)u>joxiYYmd{mN)K3rbW@tLb>mM!Ki9^)vgaCLlUc^u-pz(*W0+(?Hr?bEQNY*pQ|wX>(g?Jd$htxj{wcS30KTz%8Zbt1&fz46GJhl>8Y?K*(;wJJ&JteRliDlSLIr2w!;>xv5*3`A4Y6<{*k{jh>GfxTHpt?`6$Q?X;T5X*RV9#%Bv6HfU)RJl4;E_llJE&fV;#%6B~Y0(a+(ihmknz`OT#gHg@x=9zV+w7Ma=v50e=<2vSU@PLz;UGnfCYs|o+0+4l!QQ4Op}Of(UGjBF-W_NfFqQO+x;mgDoCRTkFO#-E2+cj_&|F;u23zGI)J&?<TL+xTyTWzFHjCUz>3lI2jV|Z;V<fAMY17Ha-q+LW_*&)7gm04I_j=yal^F@=SXRx{q-*al-?E_wQU<M6?wKHUmKhDjKbQwM%Xq>W+0|)Founx3SF=l>wQMDDp?O(bm{<+6bR`prr(rs$25y}OnW0!$6PSM6uuRP2UKDRngq=~Z_BU;m_7RhfHzhrS{_t#2+b^XQO?03Hrv#nZ%(QcQ_P+KWOSX$iUM_TW)v_n{Yq-XeE@MTO;HCMd+wD4~;ms+p9LJ>0vdan^b<XYYtC`dGqg83fE7$7P0JSNM8ZSrLacG6U8Gcg(N7;r?CMs3%<enM(@M&PGq9;Q&sruCA45uJdf>Z~pYL0e5jVr8)p2qy-jzcQT7E;%UW4&JPumE#S+@#-EE_CfW@~tNWd>fQIplo+~-W6Z3si#ST^K}d=%fQeJ7GM1p3mS}G{sRXdz!<)uwsf8ht?1IrhQs`!NLkGj+GV6pO%HfSXRjW_kc{FmQH}?4n{FWQT8dj}34=MuDWT4vxhQC%N}F^Ka4?MGX)Ac<BG?=Ig(~302iE(iZ31T{x--M9Kh0V$m3l5ED3G;+6k%k$h;t6c<)~jl52~tP$4zEEO|x=-)rr>?A2jZRHz|4iXfce*hef)DjP(&>xjeq;V_%MGNJ`Gm+uj^1EKY_%>eEZGyHMr@N~*ve5Jwqx$#tNWi44@7(i{{hX~c0-QD0oEQ3EwOrV-phSxwebcjFoK>`j-nt(0u4f-}<dj5v+&!%8>WC}ZH%9cR=%T^L<V8Bk2w+Gu^kfRTKV$3QsFjxd%TG5M8OW|FgsJ_VMJCYI}tX4}W@`h?HYU{wK!kJ@kzS(tt3$vdDY>7Mi+3q3<V=u$4ubNUN3z6mQJ+z-tX46SqMg$K~_Z$-3fmBDYMAPOZ#nR*0q!O6q74}T&gOSz63Pu1Vtf82ZV-lu;k;F1r2QBe}9JR0`(J(L*i%$j#~a_`RDxE0F1VhX4FOISd6-(z>Szc18xCh6l#-tg`}6E^SN`CEL@)&(PWK!I_liUn$l(~m*Ku<IDHqYtn}cK5e%bKPH9O_n<g6}WfjpV(3M7YUhEQEV>mvw8q#Q)uYI)q9`*L*ng&9H!)@XZJ+$IO<1&24dVgibl5X-T6<@uPhf<GC|W=Z{54|KhVaV|LArT422XXaCiTJ@yD1qP#gBU{zh=_8x8H;nB4tuGeX=a>{Pw4(WZA*jAr-Vr@z4=HokY~Lzzr(NOXZa#<uU>d1tr`*JrRFB&AAWdiT9DYXqi$^7%(0<z_x$*Tz41I{^Lz>IS{Ncjq5KAaH&Rycfj@a*jeD#2B>{WBUt^JbvAPUIpyW-{(qq-=CLAs%Y(m&X^(;p5Ye#n}zno1dU+EAkN?m1;T-DhHyMXUkPLiYdev)I{KdB)lfykk@xSSEl#g<M6>x<fJ#$$<h{%d9eLj_mj@NGX&28|H13XQu^r)-wl$CcZmj;tw~n*6fMLfq&`Q^AbA0B=JKQfh_pv9>_6bM7D=A99DNhjQm8HzlugbF#-kh=d4TaKDFE%-3>G8*b1|XhD5wjO7HsgQqd*4&rxn&$QRPPGSed`)>sp{8FkIWGoyb2~HYpn35pMP}k&Iif%>DsMCQq{L>^hnE|fBwIH{mM(%Z~ceC&1<i|a`RR&8#$L|OCV0oUE}rQ4>@s<Nlae8dIp{}Y!a?GUMhXNF2CX?v5F?YM$ThfrgFxqR(CycmbFWb8l*ZkQJrR|R`i4KPUF2fOBs*UoH`kgSj(iv&*RGV@N7J;xjsi3P{dyA*ic}vy~-EmDeoQSpCm>t@j~I7k}|sghQrY1N(z$&xxBl=Tw+}sf%OfSRZC5I>kxX|aBDhUyLhS_(r@a4Y4`SAjYUO_8)qb=Z>}{DsqUZ3`>ibdwb@|=V!py0Q-2SorF!OzQ6w#od-G(y4a;^guUw90#wWLh$bwNfZ`cUb0;Y6`=Cjvu4PxExSk~Kt<HOgh@!&i49gzCjz8BU^mrCnc>u+QsfkMoeYjr#~<*Cudb7oJ{X(=WBMrnjHr;(OrBFwFr$wpc_wUtn(Uy+P65}m8BfVIY;ta7-_0~fKh{3jF@EC=JdmN)Vabc`5kxT4j!uOG8-!>cp7HJaOi)g0VJ5r0q{<9mX+P3{3Sqdn<Lxre&}@ZoR&t8rmd20GPcYPo6+d%$b*{_n2HJN@-|WuEefy0Ni20qNI&!Qp8yIeg%Y4o~Y9Ut<ye*B_GKG|-0#(XSc*Lxt?0)TP=G{)U9It3ni-3F5?Xg^uTTrdJ-OYeFg9?T|apTjJ^~9F1QEd(72b6uH`C&~g?efshVXW&@PCPmizjmbzE6ZI*~t_W4mHTQr2j-hhq017p+KBKsg03vg!_!fdzXDW_UUHbi-XK$3pP1R&ZhXwLSlEv_o{5ax=Jgqex8oAbv|^{D0JZg9DFxuZ?-J-*ucEVjfQI&-n)4hV3oFPeeS#gh$vT5b?oQYdJUlElSHQ8rBRBF(iTPB^+;%V<BB>*$;3oqCgKI`8?{^##syhjuA)`>Ua)^BQ2+^o`FY7!OI(Kn0v{vl%4c|G6f&Eej~Q$5uj)APMCQjz%3Xro1srROgyvjvio>vz<yoO2^2uDdc|`C=})J4o-$WMF}ftg*Aa+&m=2H>H2wYo-qk6QI+`GC97J{OdTfJq^`a5jn&}ay@vCRRci(QY(KmeNEFWTa9CV!RG_VHPWazRD_oJ;>89_2J1Kq8a(u=aS`;JWe+YWsYP@#5B$ss#X{WuqQbdym7wFa$uxLPlR4>-df=EP0)!JE;UVcVV-dnH=rE){sC6&EDNs}bn4(XL|jV+*2+isOH-BT;^Z@l{s2Al$5pm$$#IL(U@ojYllp@1K#8kSEZ`b-$}{igp1Akmqu',
 'diagnostics_snapshot.py': 'c-p->ZExhpdEfObCJG9Y&Wd}w#%ZzA3b@{#kxT6OY|AYgRp4>S+2vSVk|Bp$$=d}A_(RYS1$t<Mq6m!oY8Zx-22JV~MNc34VXyul>M!Z@JTpTMxhv_M4?VyZcV3?Nw|QoEcXuC_Jd~V;teS^uI$-&9nu?6GScJ1IucV0BPoF$x(=wmKqNofPDJ$!2z(f|Mbu6+OE5cHWkcKONcXxM3%!|B~EG%a*Rq-9#9OY@sBPsH%VtT>Dyv`&q2Q22(uuf$xB55{9)w$Xi19?)+x!GiX4;cRc<2>VLOCsl~`4+kP`l!k?1*r&Sl8T7|^b~%YMIVb|DpJ0)vje0s5J?>1J{PwP*di=5V6^f^I}HBG@=|@^zmm(k%m|C0=3!jX9-lx+ej#0+MLCLT?9_6)_s5Rs(O!NL@j^29-47m@Wezg<(=e?$eS0jdaK1%i2=K$A;92ZWoh4z(!Z}}!*b)}69IsE)a@FT2-OiAhvMiS*AD>@{N>;A7!Qsg%jg<MKVq^1bA!MTV<v`dbugFOy0sp*(H_c0y)$<7|o=i<-jKcsgL9?z}Jz)D51)4@&?8Ce$+}`AdR_uRBu@Ri5CWPjB8#}aKT5W77!~S=>VwQ@{o`$qMku0qru=QYaDS74M;DAj9Ph+DTB@hs~$o&t1`E2&&T{q(kq<`#an2S<?Bf%m7<v#*9`ZF46q)8BwsA{Z$4}d3GhE+%&Id%j%d&xV{eEO4s?EoM(fvN)%QDaD{WuVp$m~oe9{Boq6BCh8}<({KF0clp%B@e<X5@P&Tm{#0F&5BB7l?=0pn}wjBN<v@&Ry>b!SjcE*s}>rw0;~hIPLt>ZJ#(;TL;{mIs`y`ZbfLuL&$y(`uIG~_b645%UL|HA8l^0VI-rA~s^{~tyma##$jtJ^hEwr#o=J2S%llw48!{)&XO)8=B0JBckYb`wD;x`q);<JRayXQ`C4UVb_(C>;a@V0D@VW7rIkeAVVHUUZ;G!iV(`zW0JEwLCQ0O*%f9XuZN>t8>IdvAEhawFpDF<R5*u-)8$I1HNAr(aHuNiR1WO9A=sbDh^W-PkC`Z7f5e3Iv9Y<_+9OL4%?)P}0QWM#<Y_0{u;y?TCq^^f@dYgWMkTjb>#Se3JCMFEmfG@k73YswoLnz<88u7QX(heessN?vV~<t-XAyKYcVFG4k<?h-)aZif^-5H5ZXy~3`ZR$l`Z07Go8S}Pvq3O$aaC{G-ohDF8WK+kEZq33Pe=Yq+zy(Ss^m|#Vf10(ZLx>^mPbWT0bm$?j616N0q6gF0<O(S3=0ts!@L_WV%s*mz{k#fmn*r0YQ-kb42CZY;jzCls<v;##6kLz?}Ycp=5GFCxj3)8US7N}~6QTAryh}~0U{sgNK+g}=qwT`2Z2jp}XrNIKqt^j3-?$lN0$kODX1xeBco<Y1LW{Dd1PempwU<@X}*-{tP>%n+mxg)Tj8uFF?Jz#(6SqNIOhB8P)pxS5!Ha3$0wiV*26|@DYF4HQ2s1rcQtSL|^v?S~|*bnX@VzPA{XhdoAst6<?0Ih+r5BkAPcLLxt+~9g1Ubuj|HI2QNE3W~(-vTD3H>*WOu@Vt&MQLK<u^rSBo_C@me*Z4g!7=^jx7SynOI=>iu3vtcu`IcM`Gq`Ckub^EpTQm+VKHokh*_?Gm8n=Ou3!FlrttXe@$mx|0qnnK>H0H<aDN$T_}R>N6q%+loYW!(C6`em&bdlu`iD+p_7Hdnmdv;eOo~3uVp@w4VcbRp`01Jo-LGY&OA8%7+Y)jfW@4&hL0fn0Fl^>1`fldN5OuaS!>Xwpf;PV>;2=T~VF_Ur(goy*5$gqF1!AS$I?79Qlal|e26`0v1H%ID!m^LH<>;HPAw=(VJb#wv6W4h|vEp^=Yh2P{zgMO2SSbcAPwnVdq+sC^oQ&IR)jFLfFgM_4%rQm)$gQ;o9n53j88-_~)U1*a<Op1F+Qoav(d5aa;fL?P&9WI7uz=kE^4E2f`{nvKS;EfOpXFT}cD&m;Ltllu71c*&ro{sM_Dm&+oVp!I$i{%868?$vc_`4TFls`A(!)|nRoXygm34(*iIhdq{`^@Trb1peBV8P3cAADWS{@}~8AN#0#a~o*M=~$z2Ocn-E%ABKiSj)RCdktim{_<7P~_&jO2YeZ{(x8kWX;b8hPUcV^2EsEl&r8d2tDkfp=`H_?z&CD+F>;R2tS2$qK(72l!NZ%Z3Phv*l;@xwd58cAQ*MDB>>tz4J(ZZPdy*%2y6wYGeZZ7S>*MSz7-*i7ihd*=g>=aoT@i(V!!KKX3jueI#d+c3K+0A-cVjZ(6>8*v3O-Kp@cJ0A-5Q|GDmOK9F7!E7}5~vI1T5MIK%)+(Na<1bt^QeTdGcA?H1YuQ5hhzgzclv8dwK*RRe1%I&^jL=;#ZMHlt|1S=SS~PBE>yr7&ll5$YGFcBwTz2Ulyu2pa>iv|spd+o_!2>&#gu=wS^($6v+5<T6k$gbB3mg}J&xwsH7Y0&0z=*?-btYonXvLFBG@O5O{S{N9-DZ=B|pq(+wG)avwl7wF!OleT!YQCSP%tnJ<`X&?!Xm}gKtKsd1g=;Jn8R$FMzct%YlT?%J--fzwzSoT6kGQdOxli=`_ldL*T!~q4_fLfUq6f&@`DLHi3+*soEA>;+9^;Q6#6<(*h)mj-*NK<M>!doGZtYAl_ymlKsp?__w-pv_6L!}~?+I(tD_do&CSMe3AWZU-Kj=r|*X^SO!Hbq@6zYd<;?cy$cR#<x-WOfu(m@)|Mug{ip_8tZt+!l*)h~_Xb-xfD8kJ;FCm;-N&pjWDRLJQF2_EI>BSJ<P%h=aKdmx2Si6iqvs(1f9FtXcq@AOL`t>Pxe$#H&Q5CI%*Uw8e0<-L&Jpl?7Zi)By6`oi%i6ERxLtY%-s=IY-dx!UNa`8B|pXm^>VaUVeq;Wxjp^1nT;;fYHwd3*i<6E)T*gfy3Tnh(#0px?~sY7ZIJpev=UZIC`0L6w9Da6)djpBNc^IYA#}d@&8e4{=U^_w<x25PKmXF<&N@nYe}kY?6nJ2Pj~6Klo;l%-mx}_ZQWzMhKj)%#`o2BWkT2pRC`X0b*Vg+pY!rua6G4;wDxlN9q=i6S#Y{RIy%JvL$3&~F?P?z?$O~KdI${VT5&nvrH(L+lJ1Tk+Loox?}tM*Wr@)_tmGvOP4f71Ife7gX#dUPf>oCl_`_Z;4uIAf_|*OH7Z(TE*xyUIm?f~xe{&_V*Seg5Px5jPZ_!W}BlB^v5V1@~`}gks!3qwIGS(|c`-mV<Ma;e%$MHcC#u)WSe*_p%?0aB0vuu>|sXPFwz&&4>>M_VyD}aN-nx^?;Z&8NDD9kPw2`~8yG_+hK;K+Lwl(>8ZziGuvi+mZ?CHVcQ$OQ(%)!|U_Zi!>+947nQM92R3ul}AbQT{%!qOhQz1f^{c;J}_PkmAUA_^7+9R|$A4pZ(D4pe*f_x*9I6+`Bq}9`?VH!(V->7Mc{h@(+haSF6R*V^cv=yXDpM_2(2TUR<)q^$wUR9cc}CEISj|SD(zGheMt3->Gvi#1i%vL~E9<Kf5$B3hDSN#J~hSpzD|a88JzWoWJ@6i{Y}Ut8HYx`rGwCUlO{?#6V#Eg)l9sSI;#CFqq>7?*X=z{uQp+uCIPxQyU*wexeTM1ycD1pvJI>wEjW>eM0FBuAN>z4;87v_a-&T{Uf^*c5|-_G<^T)5bWv5Vc%o6KCNJ&{P~Vp;K6<e2r2m{DJ9WLXfODjr6iUi{Jr`I{EMtZQ|?nDrn8y+5d6M`|B|C8kHF6p`aCWH2KF<O^$XFAzW1>Ce2BKBQ8a_l7E#|rfT$6j_Um$;+}paG?x)PXfjI(hlxKsR&oYB8H)sy~OXm0=fy3Tak$_xX=YXMpbP08$)5@vWy|Yk9Np(_oG)20B)SekkJT?bh2c?U^9Ze26xr~oGzKwX*6(5kayX#?OfHe+@t-1SCa?*ByK)p@^fNO4v2v2ZU={1HZFVgDhP{*;OV{m}ypw_eXuh@@1c=so0zc0U34)7&JHN8w3?4V<>DvM#cqKxox*gzjrjxon(tY+re!kjve^If(jBtm2}F%Y`8?_6K~6MNQhfv}#zX|N`DoC&IDvh`Q0llBQ2IJoXFfD6!P=I`vWXPR#6Lp~dII&bi#!Zzgrd+*_ZuH#_D6nEB5-H>|CcQmjGr|BfX?KGgod<t#iQ168c!M_pEDAdst{V(Cs8Rd^bFAienB8b{CG(_?&?9QT8h9&=qPE}BeVhJ|^sbp-1@k;x46G{_gzQ*WEKy{?oKy~XfO1}AR*dvRf8g2@unq%F%QLuVz^(;~>v9S5!@$plpJQVw(IvCNN!IMV>&&X`}tUX~4FpPasvt<1*s*lPhYak&~|N9zl4qpC8cF-GCVLkevuUPU8*jSSu(175$+>Pak8LvVSD_%83*u;%F$_XyW1&59}kot4Lu9F@TVOtb)(O!`8knOW~#GkOJrhz$e2rxZ=X9ph`ZBKdwJKdR+k5M0n`-|Zm?&(vXJ?qt|&%l;*fMuE9&LqsiHn09}j)MDK=Xk*RbvUGJo)mHm*zo$l8hdYg)iDibBtS@lo%?E{u;T$fvD9*dqEU%Pr_9E#?Mi^)Ob}%EFoHrgEptTP+M&N=l{KT+u4_j9!e$4e55*&Wq2t55E$k-F&){6(svp((xTGf>Sl0*n8U55%H^*00sy$;M;L9NDy_icP3Sk(JeK{|5IPSdJp|?c(A(sDBQD80PHlQJvE}Q}N|L`IbAFVWNpF9m7J$~!$2gi>e(Tk%AT=LoNF=~;~n0ldtDa3jI@q<V2Joe{!OjBXF`{={zJr3Fc(22r$n)!=NXDu`1eM^XTXa;~Pi+oEMmc666+LuM<3QIpOx&w?AUprv*`0IkPB5DVW=|SCaPWJBRvJJ02Hr2!i<k#$d_Yl)tHMCo5qxY{P)e9OpG-p#WYX+;lE+dXt=472_*c4PH#}|is_~z$K&U+s|c>Brmp9k+he){hF$I4C}>Qw|iJ8>R-_~^-TccFQ16ZB}l1!1!VWeDv6YBKrqLN^kS>K?Q$ukh#^bj;NC3QBI<Ga0%dZ0=dMzpS%WuJ1#gCJADRNT<RLW!-QMOYeD_JJ`<k>$jw<_u|nhC)-~+_W$W8W)<gW=KTRdK?{WUW)M;<@iB*&2>bW$e;?jP;GZqe6L0hgO4BHp=Js`9aAI_GEm%!Vg_$CSGko_RpE^{j6`0%gbq37|pkMbFJ4wJirBk-49~huF8%fez;F3~=dKc_JC})(=pVEvQL-h=gJLy4UfPKy&@T{FabTHtpg4*IbdwZZ;M_tK-wZ;}HH>}rOZr=k7@mQs|(jVMx-Yd2D58XEtxWw15B=8d47GwdBHKl!aT{?z@QD=Ft;otPGtgDnx<G)#8hE7)=K2HP_4k+ga0X-HA0+fFcI3xYi0+q4zf9HXZ4*',
 'export_snapshot.py': 'c-oCtOK;;g5WXAme_-7MDa2NSpt}fM_)wrJ(4yD|Xto#EAQ*8Jb62mB)Fy6_|K1srlI?gmtqwNDdGec?kDAS9*H2w*JyWb`>rNGZ&VK!Q(OGa{znJD8w{6$EAK7Psj#+r<!FlGbD!?(*;0`Kicg`Gi$|?`lE}zY2(}}60Uwe=H4#thGa@)$>jqjga+e{~=ZEMykPa>8iH~2c8jEdb(S=&A`<xuhUZjM1PH6Nr8WHp^=C^0>yC55O2dC!*93B%`m=Df{aR~cWK2As&4s$9#$j$<FsE-x;x&Oe@ATwJ}wQ#zx8eNmOs?6*Jv_&~-44elTsd35<`{5G5*Z#!rNG)1dXX~lbAF0Ob+nV}u1I^{r*1>f6-ttrF2YL#|^^2u1)T1G2^I9f4NhMk;(cOu*D4_PbIh~a1fZOc|8^7&8pzOSM2H$fsaxB@+R(>5z9bz4Z89h&A!YpIfr;EP4udp-}PHx~5j*7lGLP|IPYWEcL(65)(;v_vu0QP6gd?@8j?-WGtug94HEIpeY6a#a-2dC++f@?-~39Td2Wmd|$0jpmyrvjHv}unK-(qB0`Zw0?+1h*Dvahm3u<3L8<J9acpd2kbAZ?lm@-75uvH{4R|GInq5gN`c*u!{UbAN&}T9JOt%6XS}x+{gtLsl}s-lI}XAEo(NX1TPh!|a0AE6x0YVI{;o1^3s?$N+vgS^Zq=itW+PQvyREu>_kQqu@H%E1DsA+bEe;Lkx_<r)JA^R52QP3LTMrk6W<1f$A-oo$OOVY;ZB?oIPAg{7N2~{RW`N_kCI|RpT~PO-$AR)Y>Adp2;~T06AF9p?Whx#j2BbDa-!G4`IIY!(wpk2r$P0wpMV=ZCpR!0?lt;M4K314s5LuRQ;Ypi&<UA1?CaDYnZ~ilzThtD5SZ!Rbe6byJaK#pC6~-rzYq8#Yqgu>KWE%he2MmtsQdL$65~9qY!ayj-Ch=12^^O!zi$B^$3Pb&U$dkfH(I%;Qbyq=5_m-A9aklJsl!2gbnx`YJ;@?0j5BZ}tm_o1@;djoCXFZWO_A6aNc=tFWC;pG~W5r&vLn4L2HG(WS&p)+Dgn>(SUw2N}IfKUa7Fo#^#;krpl1FkqWBl)i?;Dho&q(zZWv4*_;wU=!Lmt>F9^~sM$a3Efm}DahB^T5`80t-?xk>k!ZTm(qsO3)@$CTe@&4kPU3Po4h?{5XdLi80e_%pIp5$*%Wmw_}BWY~(*?>fL;9vYA+XBa*i2cP;*rVk1n>?NZM<do48f->YIDIfU-tietkZ5R*7dt0S66;PJXFE4u@YE&JrQgMva{Iq@th-u|%JQKp?B8v@hxAQRGaeJWFDfVVfx|@JoP)(STKf)|=$MQA(1o<K1yd1YagfBH&tHzWl4`(<Yfy34hDb4ZF8q2o2{C0nLNCaOoC1wa;UnP^04kTsZ6e;Pxl9DH+;>X8y@-L$=LTC'}
EMBEDDED_SHA256 = {'audit_snapshot.py': '4d08e3ea1613e4248d35ebfb40716a5fafbf1a6916bfe35a99a74686d8093bc7',
 'crawler_snapshot.py': '9d32b5b911ca8ac6b53331e251e548a914281b9e794cdd87ee56451e6b28f6fa',
 'diagnostics_snapshot.py': 'aa4db8fde50a6aa965ff6f4091749334fdf7785fa6d5bf97fb9956a7fdb60acd',
 'export_snapshot.py': 'ac461b827b046ed7ea8fc4488a85d62392e4a95920ca9be7e8d714d3737a4023'}


RUN_DIR.mkdir(parents=True, exist_ok=True)
for name, encoded in EMBEDDED_FILES.items():
    target = RUN_DIR / name
    if not target.exists():
        body = zlib.decompress(base64.b85decode(encoded))
        if hashlib.sha256(body).hexdigest() != EMBEDDED_SHA256[name]:
            raise ValueError(f'Snapshot bị hỏng: {name}')
        target.write_bytes(body)
CRAWLER = RUN_DIR / 'crawler_snapshot.py'
AUDITOR = RUN_DIR / 'audit_snapshot.py'
EXPORTER = RUN_DIR / 'export_snapshot.py'
# Diagnostics chỉ đọc output, có thể cập nhật riêng mà không đổi crawler/hash resume.
diagnostic_body = zlib.decompress(base64.b85decode(EMBEDDED_FILES['diagnostics_snapshot.py']))
if hashlib.sha256(diagnostic_body).hexdigest() != EMBEDDED_SHA256['diagnostics_snapshot.py']:
    raise ValueError('Snapshot diagnostics bị hỏng.')
DIAGNOSTICS = RUN_DIR / 'diagnostics_snapshot.py'
DIAGNOSTICS.write_bytes(diagnostic_body)
diagnostics_archive = WORKDIR / f'{RUN_DIR.name}_diagnostics.zip'
if (RUN_DIR / 'crawl_config.json').exists():
    saved_config = json.loads((RUN_DIR / 'crawl_config.json').read_text(encoding='utf-8'))
    if hashlib.sha256(CRAWLER.read_bytes()).hexdigest() != saved_config['script_sha256']:
        raise ValueError('Snapshot không khớp hash của run cũ; không được ép resume.')
if not requirements_path.exists():
    requirements_path.write_text(''.join(f'{k}=={v}\n' for k, v in versions.items()), encoding='utf-8')
print('Đã chuẩn bị snapshot và ghi phiên bản dependencies.')


## 4. Tải hoặc đọc danh sách URL

Nguồn được cố định theo revision và kiểm tra SHA-256. Chỉ đọc corpus URL;
không dùng query làm nội dung hay nhãn. File Input có thể ở vị trí khác nhưng
được copy vào đường dẫn ổn định trong `/kaggle/working` để dùng lại khi resume.


In [ ]:
import requests

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

SOURCE_PATH.parent.mkdir(parents=True, exist_ok=True)
if SOURCE_PARQUET:
    supplied = Path(SOURCE_PARQUET)
    if not supplied.is_file():
        raise FileNotFoundError(supplied)
    if supplied.resolve() != SOURCE_PATH.resolve():
        shutil.copyfile(supplied, SOURCE_PATH)
elif not SOURCE_PATH.exists():
    temporary = SOURCE_PATH.with_suffix('.download')
    try:
        with requests.get(SOURCE_URL, stream=True, timeout=(15, 90)) as response:
            response.raise_for_status()
            with temporary.open('wb') as stream:
                for block in response.iter_content(1024 * 1024):
                    if block:
                        stream.write(block)
        if sha256_file(temporary) != SOURCE_SHA256:
            raise ValueError('Hash file tải về không khớp nguồn đã chốt.')
        temporary.replace(SOURCE_PATH)
    finally:
        temporary.unlink(missing_ok=True)
if sha256_file(SOURCE_PATH) != SOURCE_SHA256:
    raise ValueError('Parquet không khớp nguồn đã chốt. Kiểm tra file Input.')
print(f'Nguồn hợp lệ: {SOURCE_PATH} ({SOURCE_PATH.stat().st_size:,} byte)')


## 5. Chọn ID và tạo manifest

Nếu khoảng chọn có ID không tồn tại trong nguồn, notebook báo số lượng thiếu,
rồi lấy đúng các hàng thực sự có trong khoảng; không tạo ID hoặc URL thay thế.
Bước chuẩn bị hiện quét toàn file để kiểm tra nguồn và thống kê hostname, nên có thể mất một lúc.


In [ ]:
import pyarrow.parquet as pq
from collections import Counter
from urllib.parse import urlsplit
from IPython.display import display
from datetime import datetime, timezone
import time
import uuid

selected = pq.read_table(SOURCE_PATH, filters=[('id', '>=', ID_FROM), ('id', '<=', ID_TO)]).to_pylist()
selected.sort(key=lambda row: row['id'])
if not selected:
    raise ValueError('Không có URL nào trong khoảng ID đã chọn.')
if len({row['id'] for row in selected}) != len(selected):
    raise ValueError('Nguồn có ID trùng trong khoảng chọn.')
missing_count = ID_TO - ID_FROM + 1 - len(selected)
print(f'Chọn {len(selected):,} URL; có {missing_count:,} ID không tồn tại trong khoảng.')
print('Phân bố hostname:', dict(Counter(urlsplit(row['url']).hostname for row in selected)))
display(selected[:5])

LOG_DIR = RUN_DIR / 'logs'
LOG_DIR.mkdir(exist_ok=True)

def log_event(name, event):
    with (LOG_DIR / name).open('a', encoding='utf-8') as stream:
        stream.write(json.dumps(event, ensure_ascii=False) + '\n')

def refresh_diagnostics():
    if (RUN_DIR / 'manifest.json').exists():
        try:
            result = subprocess.run(
                [sys.executable, '-u', str(DIAGNOSTICS), '--out', str(RUN_DIR), '--archive', str(diagnostics_archive)],
                check=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                text=True, encoding='utf-8', errors='replace',
            )
            with (LOG_DIR / 'diagnostics.log').open('a', encoding='utf-8') as stream:
                stream.write(result.stdout)
        except (OSError, subprocess.CalledProcessError) as exc:
            print(f'Tạo ZIP chẩn đoán thất bại: {exc}', flush=True)

def run_script(script, *arguments):
    # Output hiển thị trực tiếp để theo dõi tiến độ; không dùng shell.
    invocation_id = uuid.uuid4().hex
    started = time.monotonic()
    operation = str(arguments[0]) if script == CRAWLER else Path(script).stem
    log_event('invocations.jsonl', {
        'event': 'start', 'invocation_id': invocation_id, 'operation': operation,
        'at': datetime.now(timezone.utc).isoformat(), 'arguments': list(map(str, arguments)),
    })
    process = subprocess.Popen(
        [sys.executable, '-u', str(script), *map(str, arguments)],
        cwd=str(WORKDIR), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding='utf-8', errors='replace', bufsize=1,
    )
    returncode = None
    try:
        with (LOG_DIR / (Path(script).stem + '.log')).open('a', encoding='utf-8') as log:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
                if operation == 'crawl':
                    try:
                        progress = json.loads(line)
                    except ValueError:
                        progress = {}
                    if isinstance(progress, dict) and 'completed_this_invocation' in progress:
                        log_event('progress.jsonl', {
                            **progress, 'invocation_id': invocation_id,
                            'at': datetime.now(timezone.utc).isoformat(),
                            'elapsed_seconds': time.monotonic() - started,
                        })
                        refresh_diagnostics()
        returncode = process.wait()
    except BaseException:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    finally:
        process.stdout.close()
        log_event('invocations.jsonl', {
            'event': 'end', 'invocation_id': invocation_id, 'operation': operation,
            'at': datetime.now(timezone.utc).isoformat(),
            'elapsed_seconds': time.monotonic() - started,
            'returncode': process.returncode,
        })
        refresh_diagnostics()
    if returncode:
        raise subprocess.CalledProcessError(returncode, process.args)

manifest_path = RUN_DIR / 'manifest.json'
if not manifest_path.exists():
    run_script(CRAWLER, 'prepare', '--source', SOURCE_PATH, '--out', RUN_DIR,
               '--size', len(selected), '--id-start', ID_FROM, '--id-stop', ID_TO + 1)
else:
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    if manifest['source_sha256'] != SOURCE_SHA256 or manifest['selected'] != len(selected):
        raise ValueError('Manifest không khớp nguồn hoặc số URL đã chọn.')
    if Path(manifest['source']).resolve() != SOURCE_PATH.resolve():
        raise ValueError('Đường dẫn nguồn của run cũ khác notebook này; dùng run mới.')
    sample_bytes = (RUN_DIR / 'sample.jsonl').read_bytes()
    if hashlib.sha256(sample_bytes).hexdigest() != manifest['sample_sha256']:
        raise ValueError('Hash sample của run cũ đã thay đổi.')
    with (RUN_DIR / 'sample.jsonl').open(encoding='utf-8') as stream:
        sample = [json.loads(line) for line in stream if line.strip()]
    if {r['id']: r['url'] for r in sample} != {r['id']: r['url'] for r in selected} or len(sample) != len(selected):
        raise ValueError('ID/URL của run cũ không khớp khoảng chọn.')
    print('Manifest hợp lệ; giữ checkpoint để resume.')


## 6. Crawl — chạy lại ô này để tiếp tục

Tiến độ in mỗi 25 URL hoàn tất hoặc khi hết lượt. Crawler tuân thủ robots,
giới hạn một request cùng registered domain, có retry và checkpoint theo ID.
`WORKERS = 4` không có nghĩa bốn request đồng thời đến cùng website.
ID đã có checkpoint, kể cả thất bại, sẽ được bỏ qua khi resume.
ZIP `<run>_diagnostics.zip` được cập nhật sau mỗi log tiến độ và khi lệnh kết thúc,
kể cả ngắt ô crawl. File chỉ chứa chẩn đoán gọn, không nén lại toàn bộ raw.


In [ ]:
arguments = ['crawl', '--out', RUN_DIR, '--workers', WORKERS,
             '--delay', DELAY, '--attempts', ATTEMPTS]
if MAX_NEW is not None:
    arguments += ['--max-new', MAX_NEW]
print('ZIP chẩn đoán tự cập nhật:', diagnostics_archive)
run_script(CRAWLER, *arguments)


## 7. Báo cáo, kiểm tra và xuất corpus

Corpus accepted chỉ gồm tài liệu qua heuristic; chưa chứng minh chất lượng bằng chấm thủ công.
Audit chỉ chạy khi đủ checkpoint. Nếu đang crawl thử, vẫn xuất phần accepted hiện có;
manifest notebook ghi rõ đây là corpus chưa hoàn tất.


In [ ]:
run_script(CRAWLER, 'report', '--out', RUN_DIR)
summary = json.loads((RUN_DIR / 'quality_report.json').read_text(encoding='utf-8'))
display({k: summary[k] for k in ['selected', 'completed', 'pending', 'accepted', 'review', 'rejected', 'status_counts']})
if summary['pending'] == 0:
    run_script(AUDITOR, '--out', RUN_DIR)
else:
    print('Run chưa hoàn tất; bỏ qua audit toàn mẫu. Chạy lại ô crawl để tiếp tục.')
if summary['accepted'] > 0:
    run_script(EXPORTER, '--out', RUN_DIR)
    corpus = pq.read_table(RUN_DIR / 'corpus.parquet')
    print(corpus.schema)
    display(corpus.select(['id', 'url', 'title']).slice(0, 5).to_pylist())
else:
    print('Chưa có tài liệu accepted; xem quarantine.jsonl và QUALITY_REPORT.md.')


## 8. Đóng gói để tải về

ZIP chứa cả raw, records, snapshot code và corpus để truy vết/resume; có thể khá lớn.
ZIP riêng `<run>_diagnostics.zip` nhỏ hơn: giải nén và mở `overview.html` để xem
tiến độ, tốc độ/ETA, HTTP status, lỗi và cờ chất lượng. Có CSV từng ID và log cài dependencies.
Tải file trong Output hoặc qua link bên dưới. Khi đổi session, cần giữ output hoặc
thêm run đã giải nén làm Kaggle Input và đặt `RESUME_RUN_PATH` trước khi chạy lại.
Resume không tự thử lại ID lỗi đã có checkpoint.


In [ ]:
from IPython.display import FileLink

notebook_manifest = {
    'id_from_inclusive': ID_FROM, 'id_to_inclusive': ID_TO,
    'dataset_revision': DATASET_REVISION, 'source_sha256': SOURCE_SHA256,
    'selected': summary['selected'], 'completed': summary['completed'],
    'pending': summary['pending'], 'accepted': summary['accepted'],
    'complete': summary['pending'] == 0,
    'packages': versions,
    'exported_files': [name for name in ['corpus.parquet', 'corpus.jsonl'] if (RUN_DIR / name).exists()],
}
(RUN_DIR / 'notebook_manifest.json').write_text(json.dumps(notebook_manifest, ensure_ascii=False, indent=2), encoding='utf-8')
refresh_diagnostics()
archive = shutil.make_archive(str(WORKDIR / RUN_DIR.name), 'zip', root_dir=RUN_DIR.parent, base_dir=RUN_DIR.name)
print('ZIP:', archive)
print('Corpus:', RUN_DIR / 'corpus.parquet')
display(FileLink(str(Path(archive).relative_to(WORKDIR))))
print('ZIP chẩn đoán:', diagnostics_archive)
display(FileLink(str(diagnostics_archive.relative_to(WORKDIR))))
if (RUN_DIR / 'corpus.parquet').exists():
    display(FileLink(str((RUN_DIR / 'corpus.parquet').relative_to(WORKDIR))))
    display(FileLink(str((RUN_DIR / 'corpus.jsonl').relative_to(WORKDIR))))
